# ECON 4370 — Homework 3: Build Your Own AI Economic Data Agent

**Due:** (instructor fills)  
**Submission:** Upload this `.ipynb` to Blackboard.

## What you will build
A **tool-using AI agent** that:
1. turns a natural-language question into a **JSON plan**,
2. executes **Python tools** (data + analysis + regression),
3. returns a short **narrative interpretation**.

You will demonstrate your agent on 4 required prompts (see the bottom of this notebook).

---

## Academic integrity + API keys
- **Do not** hard-code keys.
- **Do not** commit keys to GitHub.
- Use the input cell below.


## 0) Setup — install/import packages

In [ ]:
# If you run into missing packages, uncomment and run the next line:
# !pip -q install pandas numpy plotly statsmodels fredapi python-dotenv openai

import os
import json
import time
import re
from dataclasses import dataclass
from typing import Dict, Any, Optional, List, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf


## 1) Enter your OpenAI API key (required)

Run this cell **every time** you restart the kernel.

In [ ]:
# --- REQUIRED: Enter your OpenAI API key securely ---
# Tip: you can paste the key, hit enter, and it will be stored in-memory for this session.

from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key (input hidden): ")

# Quick check (do not print the full key)
key = os.environ.get("OPENAI_API_KEY", "")
print("Key loaded:", "YES" if key and len(key) > 20 else "NO")


## 2) FRED API key (optional but recommended)

If you have a FRED key, add it here for higher reliability. If not, your code can still work using `fredapi` without a key in many environments, but it may rate-limit.

In [ ]:
from getpass import getpass

if not os.environ.get("FRED_API_KEY"):
    tmp = getpass("Optional: paste your FRED API key (or just press Enter to skip): ")
    if tmp.strip():
        os.environ["FRED_API_KEY"] = tmp.strip()

print("FRED key loaded:", "YES" if os.environ.get("FRED_API_KEY") else "NO")


## 3) Approved series whitelist + concept mapping (required)

Your agent **must** only use series in this whitelist. If a user asks for something else, it must refuse gracefully and suggest allowed alternatives.

In [ ]:
# --- REQUIRED: whitelist of allowed FRED series IDs ---
# You may expand this, but keep it a *whitelist* (approved list).
SERIES_WHITELIST: Dict[str, str] = {
    "CPIAUCSL": "CPI (All Urban Consumers) - price level (monthly)",
    "CPILFESL": "Core CPI (All Urban Consumers, less food & energy) - price level (monthly)",
    "UNRATE": "Unemployment Rate (monthly, %)",
    "FEDFUNDS": "Effective Federal Funds Rate (monthly, %)",
    "DCOILWTICO": "Crude Oil Prices: WTI (daily, $/barrel)",
    # Choose a wage series (example):
    "CES0500000003": "Average Hourly Earnings: Total Private (monthly, $)",
}

# --- REQUIRED: concept -> series mapping layer ---
# These are the words a user might type; you map them to series IDs.
CONCEPT_TO_SERIES: Dict[str, str] = {
    "inflation": "CPIAUCSL",
    "cpi": "CPIAUCSL",
    "core inflation": "CPILFESL",
    "core cpi": "CPILFESL",
    "unemployment": "UNRATE",
    "unemployment rate": "UNRATE",
    "fed funds": "FEDFUNDS",
    "federal funds rate": "FEDFUNDS",
    "oil": "DCOILWTICO",
    "oil price": "DCOILWTICO",
    "wages": "CES0500000003",
    "wage": "CES0500000003",
}

def is_allowed_series(series_id: str) -> bool:
    return series_id in SERIES_WHITELIST

def suggest_allowed_options(user_text: str, k: int = 5) -> List[str]:
    # lightweight suggestions (string overlap)
    user_text = user_text.lower()
    scored = []
    for sid, desc in SERIES_WHITELIST.items():
        score = sum(tok in (sid.lower() + " " + desc.lower()) for tok in user_text.split())
        scored.append((score, sid, desc))
    scored.sort(reverse=True)
    return [f"{sid} — {desc}" for score, sid, desc in scored[:k]]


## 4) FRED data helper (required)

You must handle frequency mismatch (e.g., daily oil to monthly) before merging.

In [ ]:
# --- FRED data pull helper ---
# Using fredapi (installed in many class environments); alternatively you can use requests.

from fredapi import Fred

def fred_series(series_id: str, start: str = "1990-01-01", end: Optional[str] = None) -> pd.Series:
    if not is_allowed_series(series_id):
        raise ValueError(f"Series '{series_id}' is not in whitelist.")
    fred = Fred(api_key=os.environ.get("FRED_API_KEY"))
    s = fred.get_series(series_id, observation_start=start, observation_end=end)
    s.index = pd.to_datetime(s.index)
    s.name = series_id
    return s

def to_monthly(s: pd.Series, how: str = "mean") -> pd.Series:
    """Convert a Series to monthly frequency.
    - If already monthly-ish, keep.
    - If daily/weekly, resample to month using mean or last.
    """
    s = s.dropna().copy()
    # Heuristic: if there are many observations per month, treat as high-frequency
    # We'll resample anything with median diff < 20 days to monthly.
    if len(s) >= 3:
        med_days = np.median(np.diff(s.index.values).astype('timedelta64[D]').astype(int))
    else:
        med_days = 31

    if med_days < 20:
        if how == "last":
            out = s.resample("M").last()
        else:
            out = s.resample("M").mean()
    else:
        # Make sure it's end-of-month indexed for consistent merging
        out = s.resample("M").last()
    out.index = out.index.to_period("M").to_timestamp("M")
    return out


## 5) Your tools (required)

Implement **three tools** and make the agent call them based on the JSON plan.

In [ ]:
# --- TOOL 1: build_dataset ---
def build_dataset(series: List[str],
                  start: str = "1990-01-01",
                  end: Optional[str] = None,
                  monthly_how: str = "mean") -> pd.DataFrame:
    """Pull 2+ series, harmonize frequency to monthly, merge into one DataFrame."""
    if len(series) < 2:
        raise ValueError("build_dataset requires at least 2 series IDs.")

    frames = []
    for sid in series:
        s = fred_series(sid, start=start, end=end)
        s_m = to_monthly(s, how=monthly_how)
        frames.append(s_m)

    df = pd.concat(frames, axis=1).sort_index()
    df.index.name = "date"
    return df

# --- helper transformations ---
def yoy_pct(series: pd.Series) -> pd.Series:
    # YoY % change on a monthly series
    return 100 * (series / series.shift(12) - 1)

# --- TOOL 2: analyze_dataset ---
def analyze_dataset(df: pd.DataFrame,
                    y_col: str,
                    x_col: Optional[str] = None,
                    compute_yoy_for: Optional[str] = None,
                    since: Optional[str] = None) -> Dict[str, Any]:
    """Make a plot + compute stats. Returns dict for the agent to narrate."""
    d = df.copy()
    if since:
        d = d.loc[pd.to_datetime(since):].copy()

    result: Dict[str, Any] = {"n_obs": int(d.dropna().shape[0])}

    # optional YoY transformation
    if compute_yoy_for:
        if compute_yoy_for not in d.columns:
            raise ValueError(f"{compute_yoy_for} not in df columns.")
        d[f"{compute_yoy_for}_yoy"] = yoy_pct(d[compute_yoy_for])
        result["yoy_column"] = f"{compute_yoy_for}_yoy"

    # plot
    plot_cols = [c for c in [y_col, x_col] if c]
    fig = px.line(d, x=d.index, y=plot_cols, title=f"Series over time: {', '.join(plot_cols)}")
    fig.show()

    # stats
    result["summary"] = d[plot_cols].describe().to_dict()

    if x_col:
        corr = d[[y_col, x_col]].dropna().corr().iloc[0,1]
        result["corr"] = float(corr)

    # min/max since period
    result["minmax"] = {}
    for c in plot_cols:
        dd = d[c].dropna()
        if dd.empty:
            continue
        result["minmax"][c] = {
            "min": float(dd.min()),
            "min_date": str(dd.idxmin().date()),
            "max": float(dd.max()),
            "max_date": str(dd.idxmax().date()),
        }
    return result

# --- TOOL 3: run_regression ---
def run_regression(df: pd.DataFrame,
                   formula: str,
                   since: Optional[str] = None) -> Dict[str, Any]:
    d = df.copy()
    if since:
        d = d.loc[pd.to_datetime(since):].copy()

    model = smf.ols(formula=formula, data=d).fit()
    out = {
        "formula": formula,
        "n_obs": int(model.nobs),
        "r2": float(model.rsquared),
        "params": {k: float(v) for k, v in model.params.items()},
        "bse": {k: float(v) for k, v in model.bse.items()},
    }
    print(model.summary())
    return out


## 6) OpenAI call helper + JSON robustness (required)

Your planner must output JSON **only**. Your code must handle invalid JSON by repairing/retrying once.

In [ ]:
from openai import OpenAI
client = OpenAI()

JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def extract_json(text: str) -> str:
    m = JSON_RE.search(text)
    if not m:
        raise ValueError("No JSON object found in model output.")
    return m.group(0)

def plan_with_llm(user_prompt: str, system_prompt: str, model: str = "gpt-4.1-mini") -> Dict[str, Any]:
    """Ask the LLM for a JSON plan. Repair once if needed."""
    def _call(msgs):
        resp = client.chat.completions.create(
            model=model,
            messages=msgs,
            temperature=0.2,
        )
        return resp.choices[0].message.content

    msgs = [
        {"role":"system", "content": system_prompt},
        {"role":"user", "content": user_prompt},
    ]
    raw = _call(msgs)

    for attempt in range(2):
        try:
            js = extract_json(raw)
            return json.loads(js)
        except Exception as e:
            if attempt == 1:
                raise
            # repair prompt
            repair = (
                "Your previous response was not valid JSON. "
                "Return ONLY a valid JSON object that matches the schema, no backticks, no commentary."
            )
            msgs2 = msgs + [{"role":"assistant", "content": raw}, {"role":"user", "content": repair}]
            raw = _call(msgs2)

    raise RuntimeError("Unreachable")


## 7) Planner prompt (YOU must write/modify)

This is where you enforce:
- whitelist-only series
- tool selection
- transformations (YoY when needed)
- regression when asked

**You must customize this.**

In [ ]:
PLANNER_SYSTEM_PROMPT = f"""You are an economics data agent planner.

Rules:
- You MUST output ONLY a single JSON object. No markdown, no backticks, no extra text.
- You can only use series from this whitelist: {list(SERIES_WHITELIST.keys())}
- If the user requests a series not in the whitelist, set tool to null and explain in narrative with 2-3 allowed alternatives.

Available tools:
1) build_dataset: args={{"series":[...], "start":"YYYY-MM-DD", "end":null, "monthly_how":"mean"|"last"}}
2) analyze_dataset: args={{"y_col":"...", "x_col":null|"...", "compute_yoy_for":null|"...", "since":null|"YYYY-MM-DD"}}
3) run_regression: args={{"formula":"...", "since":null|"YYYY-MM-DD"}}

JSON schema:
{{
  "tool": "build_dataset" | "analyze_dataset" | "run_regression" | null,
  "args": {{}},
  "analysis_steps": ["..."],
  "narrative": "..."
}}

Planning guidance:
- If the user wants a comparison of two concepts/series over time, choose build_dataset first.
- If the user asks for trends/correlation/plot, choose analyze_dataset.
- If the user asks for evidence of a relationship or a 'Phillips curve', choose run_regression (after dataset exists).
- Use YoY for inflation or wage-growth questions by setting compute_yoy_for appropriately, and reference *_yoy columns in later steps.

Keep narratives short and clear.
"""

## 8) Agent loop (required)

You will implement a loop that:
1) gets a JSON plan
2) executes the tool
3) optionally continues (e.g., build_dataset → analyze/regression)

For grading, it’s fine to implement a simple 2-step pipeline:
- If plan.tool is build_dataset, build df and then ask planner again with a message like “Dataset built with columns …”.


In [ ]:
# --- REQUIRED: agent memory store ---
AGENT_STATE: Dict[str, Any] = {"df": None, "last_result": None}

def run_agent(user_question: str, model: str = "gpt-4.1-mini") -> Dict[str, Any]:
    """Simple agent runner. Extend as needed."""
    # 1) initial plan
    plan = plan_with_llm(user_question, PLANNER_SYSTEM_PROMPT, model=model)
    print("PLAN:", json.dumps(plan, indent=2))

    tool = plan.get("tool")
    args = plan.get("args", {})

    # 2) execute tool
    if tool is None:
        return {"plan": plan, "result": None}

    if tool == "build_dataset":
        df = build_dataset(**args)
        AGENT_STATE["df"] = df
        print("Dataset built. Columns:", list(df.columns), "Rows:", len(df))
        # follow-up plan: tell planner what columns exist
        followup_prompt = (
            user_question
            + "\n\nDATASET_READY: columns="
            + ", ".join(df.columns)
            + f"; date_range={df.index.min().date()} to {df.index.max().date()}"
            + "\nIf you need YoY, ask analyze_dataset to compute it."
        )
        plan2 = plan_with_llm(followup_prompt, PLANNER_SYSTEM_PROMPT, model=model)
        print("PLAN 2:", json.dumps(plan2, indent=2))
        tool2 = plan2.get("tool")
        args2 = plan2.get("args", {})
        if tool2 == "analyze_dataset":
            res = analyze_dataset(AGENT_STATE["df"], **args2)
        elif tool2 == "run_regression":
            res = run_regression(AGENT_STATE["df"], **args2)
        else:
            res = None
        AGENT_STATE["last_result"] = res
        return {"plan": plan, "plan2": plan2, "result": res}

    elif tool == "analyze_dataset":
        if AGENT_STATE["df"] is None:
            raise ValueError("No dataset in memory. Run build_dataset first or modify planner to do so.")
        res = analyze_dataset(AGENT_STATE["df"], **args)
        AGENT_STATE["last_result"] = res
        return {"plan": plan, "result": res}

    elif tool == "run_regression":
        if AGENT_STATE["df"] is None:
            raise ValueError("No dataset in memory. Run build_dataset first or modify planner to do so.")
        res = run_regression(AGENT_STATE["df"], **args)
        AGENT_STATE["last_result"] = res
        return {"plan": plan, "result": res}

    else:
        raise ValueError(f"Unknown tool: {tool}")


## 9) Required Demo Runs (graded)

Run your agent on each prompt below. Your output must show:
- JSON plan(s)
- tool execution
- plot(s) where relevant
- narrative interpretation

**Prompts:**
1. Is there evidence of a Phillips Curve since 1990?
2. Compare oil prices and inflation after 2015.
3. Did wages keep up with inflation after COVID?
4. Compare federal funds rate and inflation since 2000.


In [ ]:
# REQUIRED DEMO RUN 1
out1 = run_agent("Is there evidence of a Phillips Curve since 1990?")
out1

In [ ]:
# REQUIRED DEMO RUN 2
out2 = run_agent("Compare oil prices and inflation after 2015.")
out2

In [ ]:
# REQUIRED DEMO RUN 3
out3 = run_agent("Did wages keep up with inflation after COVID?")
out3

In [ ]:
# REQUIRED DEMO RUN 4
out4 = run_agent("Compare federal funds rate and inflation since 2000.")
out4

## 10) Notes (required)

Write a short reflection:
- What series are in your whitelist and why?
- How did you handle frequency mismatch?
- What did you do when the planner produced invalid JSON?


**Your notes (write below):**

- Whitelist rationale:
- Frequency mismatch approach:
- JSON robustness approach:
